# Example 03 — Two-DOF Chain with Unilateral Spring (Impact Dynamics)

Frequency-response and phase-portrait analysis of a 2-DOF chain with a unilateral (contact) spring at DOF 1, capturing the asymmetric impact nonlinearity via high-harmonic HB. (Krack & Gross 2019)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.nonlinearities.elements import unilateral_spring
from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions
from nlvib.visualization.plots import plot_frf, plot_phase_portrait

In [ ]:
# --- System setup ---
MASSES = [1.0, 1.0]
STIFFNESSES = [1.0, 0.5, 1.0]
DAMPINGS = [0.02, 0.02, 0.02]
CONTACT_STIFFNESS = 5.0
CONTACT_GAP = 0.1
CONTACT_DOF = 1
EXCITATION_DOF = 0
EXCITATION_AMPLITUDE = 0.1
N_HARMONICS = 7  # unilateral springs need more harmonics
OMEGA_START = 0.5
OMEGA_END = 1.8
NEWTON_TOL = 1e-10

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
system.add_nonlinear_element(unilateral_spring(k=CONTACT_STIFFNESS, gap=CONTACT_GAP, dof_index=CONTACT_DOF))
print(f'System: {system}')

In [ ]:
# --- Initial solution at OMEGA_START ---
n_dof = system.n_dof
n_total = n_dof * (2 * N_HARMONICS + 1)
excitation = {'dof': EXCITATION_DOF, 'amplitude': EXCITATION_AMPLITUDE}
Q0 = np.zeros(n_total)
for _ in range(50):
    R, J = hb_residual(Q0, OMEGA_START, system, N_HARMONICS, excitation)
    if np.linalg.norm(R) < NEWTON_TOL: break
    try: dQ = np.linalg.solve(J, -R)
    except np.linalg.LinAlgError: dQ = np.linalg.lstsq(J, -R, rcond=None)[0]
    Q0 = Q0 + dQ
print(f'Initial residual norm: {np.linalg.norm(R):.3e}')

In [ ]:
# --- Run HB continuation ---
def residual_fn(Q, omega):
    return hb_residual(Q, omega, system, N_HARMONICS, excitation)

opts = ContinuationOptions(
    ds_initial=0.02, ds_min=1e-5, ds_max=0.1, max_steps=600,
    max_newton_iter=25, newton_tol=NEWTON_TOL, adapt_step=True,
    lambda_min=OMEGA_START, lambda_max=OMEGA_END,
)
result = ContinuationSolver().run(residual_fn, Q0, OMEGA_START, opts)
print(f'Continuation: {result.n_steps} steps, {result.message}')

In [ ]:
# --- Extract amplitudes and save plots ---
solutions = result.solutions
omega_branch = solutions[:, -1]

amplitudes = np.zeros((n_dof, len(omega_branch)))
for i in range(len(omega_branch)):
    Q = solutions[i, :n_total]
    for d in range(n_dof):
        c_idx = (2*1-1)*n_dof + d
        s_idx = 2*1*n_dof + d
        amplitudes[d, i] = np.sqrt(Q[c_idx]**2 + Q[s_idx]**2)

stability_for_plot = ~result.stability

class FRFResult:
    def __init__(self, omega, amplitude, stability):
        self.omega = omega; self.amplitude = amplitude; self.stability = stability

output_dir = Path('..') / 'examples' / '03_two_dof_unilateral' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

frf_result = FRFResult(omega=omega_branch, amplitude=amplitudes, stability=stability_for_plot)
fig_frf = plot_frf(frf_result, dof=CONTACT_DOF, harmonic=1)
fig_frf.suptitle(f'Example 03 — Two-DOF unilateral spring\nDOF {CONTACT_DOF} (contact side), H={N_HARMONICS}', fontsize=9)
fig_frf.savefig(output_dir / 'frf.png', dpi=150, bbox_inches='tight')
print('Saved frf.png')

# Phase portrait at peak
amp_dof1 = amplitudes[CONTACT_DOF, :]
peak_idx = int(np.argmax(amp_dof1))
omega_peak = float(omega_branch[peak_idx])
Q_peak = solutions[peak_idx, :-1]

T = 2.0 * np.pi / omega_peak
n_time = 512
t_ts = np.linspace(0, 2*T, n_time, endpoint=False)
Q_mat = Q_peak.reshape(2*N_HARMONICS+1, n_dof).T
q_time = np.zeros((n_dof, n_time))
dq_time = np.zeros((n_dof, n_time))
q_time += Q_mat[:, 0:1]
for h in range(1, N_HARMONICS+1):
    hw = h * omega_peak
    c_idx, s_idx = 2*h-1, 2*h
    q_time  += Q_mat[:, c_idx:c_idx+1]*np.cos(hw*t_ts) + Q_mat[:, s_idx:s_idx+1]*np.sin(hw*t_ts)
    dq_time += hw*Q_mat[:, s_idx:s_idx+1]*np.cos(hw*t_ts) - hw*Q_mat[:, c_idx:c_idx+1]*np.sin(hw*t_ts)

n_pts = n_time // 2
fig_phase = plot_phase_portrait(t_ts[n_pts:], q_time[:, n_pts:], dq_time[:, n_pts:], dof=CONTACT_DOF)
fig_phase.suptitle(f'Example 03 — Phase portrait at peak\nDOF {CONTACT_DOF}, omega={omega_peak:.4f} rad/s', fontsize=9)
fig_phase.savefig(output_dir / 'phase_portrait.png', dpi=150, bbox_inches='tight')
print('Saved phase_portrait.png')

In [ ]:
# --- Display FRF inline ---
fig_frf

In [ ]:
# --- Display phase portrait inline ---
fig_phase

In [ ]:
# --- Summary ---
amp_peak = float(amp_dof1[peak_idx])
print('=' * 60)
print('SUMMARY — Example 03: Two-DOF Unilateral Spring')
print('=' * 60)
print(f'  Peak amplitude (DOF {CONTACT_DOF}, H=1): {amp_peak:.6f} m')
print(f'  Frequency at peak:                       {omega_peak:.6f} rad/s')
print(f'  Total continuation steps:                {result.n_steps}')
print(f'  Omega range covered:                     [{omega_branch.min():.4f}, {omega_branch.max():.4f}]')
print('=' * 60)